# HW1 EDA — HeadBook Influence Maximization

Exploratory plots that are NOT part of the submission (uses matplotlib, which is not in the allowed-imports list for the submitted .py).

Track C from `plan.md`:
1. Network topology (degree distribution, fat tail).
2. Risk vs. reward scatter (cost vs. boomerang risk).
3. Community visualization of the candidate pool.
4. Penalty sweep plot (after running `tune_penalty` from the shield).

Reads results from `solution_output.csv` (written by the shield's `main()`).

In [ ]:
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

import campaign_simulation as cs

graph = cs.read_graph()
costs = cs.read_costs()
probabilities = cs.read_probabilities()

print(f'Nodes: {graph.number_of_nodes()}, Edges: {graph.number_of_edges()}')

## 1. Network topology

In [ ]:
degrees = np.array([d for _, d in graph.degree()])
print(f'Degree: min={degrees.min()} mean={degrees.mean():.1f} median={np.median(degrees):.0f} max={degrees.max()}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(degrees, bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Degree distribution (linear)')
axes[0].set_xlabel('degree'); axes[0].set_ylabel('count')

axes[1].hist(degrees, bins=np.logspace(0, np.log10(degrees.max()+1), 40), color='darkorange', edgecolor='white')
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_title('Degree distribution (log-log)')
axes[1].set_xlabel('degree'); axes[1].set_ylabel('count')
plt.tight_layout(); plt.show()

In [ ]:
# Top-20 by degree and by degree/cost
rows = []
for node in graph.nodes():
    rows.append({'user_id': node, 'degree': graph.degree(node), 'cost': costs[node]})
df = pd.DataFrame(rows)
df['deg_per_cost'] = df['degree'] / df['cost']

print('Top 20 by degree:')
print(df.nlargest(20, 'degree').to_string(index=False))
print('\nTop 20 by degree/cost:')
print(df.nlargest(20, 'deg_per_cost').to_string(index=False))

## 2. Risk vs. reward scatter

In [ ]:
rows = []
for node in graph.nodes():
    p = probabilities[node]
    rows.append({
        'user_id': node,
        'cost': costs[node],
        'degree': graph.degree(node),
        'risk': p['p_minus'] + p['a_plus'],
        'reward': p['p_plus'] + p['a_minus'],
    })
risk_df = pd.DataFrame(rows)

# Load final selection if available
selected_set = set()
if os.path.exists('solution_output.csv'):
    sel = cs.read_influencers_from_csv('solution_output.csv', costs)
    if sel is not None:
        selected_set = set(sel)
        print(f'Highlighting {len(selected_set)} selected influencers.')

is_sel = risk_df['user_id'].isin(selected_set)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(risk_df.loc[~is_sel, 'cost'], risk_df.loc[~is_sel, 'risk'],
           s=risk_df.loc[~is_sel, 'degree']*0.5 + 2, alpha=0.25, c='lightgray', label='all users')
ax.scatter(risk_df.loc[is_sel, 'cost'], risk_df.loc[is_sel, 'risk'],
           s=risk_df.loc[is_sel, 'degree']*0.5 + 10, alpha=0.9, c='red', label='selected', edgecolor='black')
ax.set_xlabel('cost'); ax.set_ylabel('risk = p_minus + a_plus')
ax.set_title('Risk vs. cost (marker size = degree)')
ax.legend(); plt.tight_layout(); plt.show()

## 3. Community map of the candidate pool

In [ ]:
features = cs.compute_node_features(graph, costs, probabilities, penalty_w=cs.PENALTY_W)
pool = cs.build_candidate_pool(features, pool_size=cs.CANDIDATE_POOL_SIZE)
communities = cs.detect_communities(graph)

sub = graph.subgraph(pool)
print(f'Pool subgraph: {sub.number_of_nodes()} nodes, {sub.number_of_edges()} edges')

pos = nx.spring_layout(sub, seed=42, k=0.5)
labels = np.array([communities.get(n, -1) for n in sub.nodes()])
node_sel = ['red' if n in selected_set else 'lightgray' for n in sub.nodes()]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
nx.draw_networkx(sub, pos=pos, ax=axes[0], node_color=labels, cmap='tab20',
                 with_labels=False, node_size=80, edge_color='lightgray', alpha=0.85)
axes[0].set_title('Candidate pool colored by community')
axes[0].axis('off')

nx.draw_networkx(sub, pos=pos, ax=axes[1], node_color=node_sel,
                 with_labels=False, node_size=80, edge_color='lightgray', alpha=0.85)
axes[1].set_title('Candidate pool — red = chosen, gray = not chosen')
axes[1].axis('off')
plt.tight_layout(); plt.show()

## 4. Penalty sweep plot

Run `cs.tune_penalty(graph, costs, probabilities, trials_per=30)` once (slow!) and save the resulting DataFrame to `penalty_sweep.csv`, then plot.

In [ ]:
# Uncomment to actually run the sweep (takes a long time):
sweep_df = cs.tune_penalty(graph, costs, probabilities,
                           weights=[1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
                           trials_per=30)
sweep_df.to_csv('penalty_sweep.csv', index=False)

if os.path.exists('penalty_sweep.csv'):
    sweep_df = pd.read_csv('penalty_sweep.csv').sort_values('penalty_w')
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(sweep_df['penalty_w'], sweep_df['mean'], yerr=sweep_df['ci95'],
                marker='o', capsize=4, color='steelblue')
    ax.set_xlabel('PENALTY_W'); ax.set_ylabel('mean net support')
    ax.set_title('Mean net support vs. PENALTY_W (error bars = 95% CI)')
    ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
    print('\nRanked by mean - 0.25*std:')
    print(sweep_df.sort_values('score_adj', ascending=False).to_string(index=False))
else:
    print('penalty_sweep.csv not found — uncomment the cell above to generate it.')